## `sudo` deep dive — `/etc/sudoers` and `visudo`

`sudo`'s power comes from its policy file, `/etc/sudoers` (plus drop-ins in `/etc/sudoers.d/`). It's a small language of its own.

**Never edit `/etc/sudoers` directly — always use `visudo`.** It checks the syntax before saving; a broken sudoers can lock you out of all administration.

```bash
sudo visudo                            # edit the main file
sudo visudo -f /etc/sudoers.d/myrule   # edit a drop-in (preferred — survives package updates)
```

Every rule has the shape **WHO  WHERE = (AS_WHOM)  WHAT**:

```sudoers
alice  ALL=(ALL:ALL) ALL                          # alice: any command as anyone, with her password
%sudo  ALL=(ALL:ALL) ALL                           # the default: the sudo group can do anything
alice  ALL=(root) NOPASSWD: /bin/systemctl restart nginx   # one command, no password
%devs  ALL=(root) NOPASSWD: /usr/bin/journalctl    # a group, one command
```

- **WHO** — a username, or **`%group`** for a group.
- **WHERE** — hosts the rule applies to (`ALL` unless you share one sudoers across machines).
- **(AS_WHOM)** — which identity sudo may switch to; defaults to root.
- **WHAT** — full command paths, comma-separated (`ALL` = anything).
- **`NOPASSWD:`** — skip the password prompt for those commands.

Prefer **drop-in files** in `/etc/sudoers.d/` over editing the main file — they keep your rules separate from what package updates rewrite. And the first debugging step whenever "sudo isn't working": **`sudo -l`**, which lists exactly what you're allowed to run.
